In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Verification Gate
input_file = "master_feature_table.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Missing master feature bank! Please run Notebook 05A first.")

master = pd.read_csv(input_file)
print("=====================================================================")
print(f"✅ SUCCESS: Ingested Consolidated Table for Glass-Box Hazard Decomposition.")
print("=====================================================================")
print(f" -> Current Matrix Shape: {master.shape[0]} rows x {master.shape[1]} columns")

✅ SUCCESS: Ingested Consolidated Table for Glass-Box Hazard Decomposition.
 -> Current Matrix Shape: 18260 rows x 45 columns


## The Glass-Box Hazard Decomposer Function

In [3]:
def decompose_trip_hazards(row):
    """
    Computes and returns the independent sub-hazard scores and their 
    underlying mathematical point contributions for complete glass-box explainability.
    """
    # -----------------------------------------------------------------
    # 1. WEATHER HAZARD CHANNEL (Max 100)
    # -----------------------------------------------------------------
    rain_comp = min((row['rain'] / 50.0) * 50.0, 50.0)
    snow_comp = min((row['snowfall'] / 20.0) * 30.0, 30.0)
    wind_comp = min((row['wind_speed'] / 40.0) * 20.0, 20.0)
    weather_hazard = min(rain_comp + snow_comp + wind_comp, 100.0)
    
    # -----------------------------------------------------------------
    # 2. LANDSLIDE HAZARD CHANNEL (Max 100)
    # -----------------------------------------------------------------
    distance_factor = 100.0 / (row['nearest_landslide_km'] + 1.0)
    density_factor = row['landslide_density_per_1000sqkm'] * 50.0
    terrain_factor = row['mountain_flag'] * 20.0
    
    dist_comp = distance_factor * 0.4
    dens_comp = density_factor * 0.3
    terr_comp = terrain_factor * 0.3
    landslide_hazard = min(dist_comp + dens_comp + terr_comp, 100.0)
    
    # -----------------------------------------------------------------
    # 3. CROWD HAZARD CHANNEL (Max 100)
    # -----------------------------------------------------------------
    crowd_base_comp = row['crowd_baseline']
    festival_comp = row['festival_boost']
    vacation_comp = row['school_vacation_flag'] * 10.0
    crowd_hazard = min(crowd_base_comp + festival_comp + vacation_comp, 100.0)
    
    # -----------------------------------------------------------------
    # 4. TRANSPORT HAZARD CHANNEL (Max 100)
    # -----------------------------------------------------------------
    complexity_comp = row['transport_complexity_score'] * 0.5
    budget_comp = row['budget_stress_index'] * 0.3
    elevation_comp = row['elevation_penalty'] * 0.2
    transport_hazard = min(complexity_comp + budget_comp + elevation_comp, 100.0)
    
    # -----------------------------------------------------------------
    # 5. OVERALL COMPOSITE HAZARD BLEND
    # -----------------------------------------------------------------
    overall_hazard = (
        (weather_hazard * 0.35) + 
        (landslide_hazard * 0.25) + 
        (crowd_hazard * 0.20) + 
        (transport_hazard * 0.20)
    )
    
    return pd.Series([
        weather_hazard, landslide_hazard, crowd_hazard, transport_hazard, overall_hazard,
        rain_comp, snow_comp, wind_comp,
        dist_comp, dens_comp, terr_comp,
        crowd_base_comp, festival_comp, vacation_comp,
        complexity_comp, budget_comp, elevation_comp
    ])

print("Explainable glass-box risk calculation matrix compiled successfully.")

Explainable glass-box risk calculation matrix compiled successfully.


## Engine Execution, Percentile Ranking, & Storage

In [4]:
# 1. Declare target column receiver grid
engine_columns = [
    'weather_hazard_score', 'landslide_hazard_score', 'crowd_hazard_score', 'transport_hazard_score', 'overall_hazard_score',
    'rain_hazard_component', 'snow_hazard_component', 'wind_hazard_component',
    'distance_hazard_component', 'density_hazard_component', 'terrain_hazard_component',
    'crowd_baseline_component', 'festival_component', 'vacation_component',
    'complexity_component', 'budget_component', 'elevation_component'
]

In [5]:
# Map decomposer function over rows
master[engine_columns] = master.apply(decompose_trip_hazards, axis=1)

In [6]:
# 2. UPGRADE: HIGH-SPEED VECTORIZED RISK SHARE CALCULATION
# Calculate the raw absolute cumulative hazard sum across columns instantly
total_hazard_sum = (
    master['weather_hazard_score'] + master['landslide_hazard_score'] + 
    master['crowd_hazard_score'] + master['transport_hazard_score']
)

In [7]:
# Use a safe denominator array to prevent division-by-zero crashes
safe_denominator = np.where(total_hazard_sum == 0, 1.0, total_hazard_sum)

In [8]:
# Broadcast shares vectorially across all 18,260 rows instantly
master['weather_risk_share'] = np.where(total_hazard_sum == 0, 0.25, master['weather_hazard_score'] / safe_denominator)
master['landslide_risk_share'] = np.where(total_hazard_sum == 0, 0.25, master['landslide_hazard_score'] / safe_denominator)
master['crowd_risk_share'] = np.where(total_hazard_sum == 0, 0.25, master['crowd_hazard_score'] / safe_denominator)
master['transport_risk_share'] = np.where(total_hazard_sum == 0, 0.25, master['transport_hazard_score'] / safe_denominator)

In [9]:
# Round all share percentages to clean 4-decimal fractions
share_cols = ['weather_risk_share', 'landslide_risk_share', 'crowd_risk_share', 'transport_risk_share']
for col in engine_columns + share_cols:
    master[col] = master[col].round(4)

In [10]:
# 3. Macro Rank Percentile & Bucketing
master['risk_percentile'] = (master['overall_hazard_score'].rank(pct=True) * 100.0).round(2)

In [11]:
def assign_risk_category(score):
    if score < 25.0:  return "Low"
    if score < 50.0:  return "Moderate"
    if score < 75.0:  return "High"
    return "Extreme"

master['risk_category'] = master['overall_hazard_score'].apply(assign_risk_category)

In [12]:
# Drop old target placeholders if any exist
if 'trip_risk_score' in master.columns:
    master = master.drop(columns=['trip_risk_score'])

In [13]:
# Export the fully transparent master training asset to disk
output_file = "master_feature_table_with_hazards.csv"
master.to_csv(output_file, index=False)

In [14]:
print(f"\n🎉 SUCCESS: Explainable target matrix + vectorized shares written to '{output_file}'")
print(master['risk_category'].value_counts())


🎉 SUCCESS: Explainable target matrix + vectorized shares written to 'master_feature_table_with_hazards.csv'
risk_category
Moderate    9913
Low         8304
High          43
Name: count, dtype: int64


## Production Quality Gate & Share Additivity Assertions

In [15]:
print("=====================================================================")
print("🏁 GLASS-BOX INFRASTRUCTURE QUALITY ASSURANCE REPORT")
print("=====================================================================")

# Check 1: Enforce bounding limits across main scores
score_checks = ['weather_hazard_score', 'landslide_hazard_score', 'crowd_hazard_score', 'transport_hazard_score', 'overall_hazard_score', 'risk_percentile']
for score_col in score_checks:
    assert master[score_col].between(0.0, 100.0).all(), f"QA Failure: Out of bounds values inside {score_col}"

# Check 2: Verify that shares sum up to exactly 100% (1.0) on every row
computed_share_totals = master['weather_risk_share'] + master['landslide_risk_share'] + master['crowd_risk_share'] + master['transport_risk_share']
assert np.allclose(computed_share_totals, 1.0, atol=1e-4), "QA Failure: Risk shares do not add up to 1.0 (100%) additivity."

🏁 GLASS-BOX INFRASTRUCTURE QUALITY ASSURANCE REPORT


In [16]:
print(" ✅ ALL PROGRAMMATIC BOUNDARY AND SHARE ADDITIVITY ASSERTIONS PASSED!")
print("     Risk metrics and pie-chart share vectors are certified 100% sound.")

print("\n=====================================================================")
print("📊 PORTFOLIO PREVIEW: VERIFYING DYNAMIC SHARE RATIOS")
print("=====================================================================")
# Display a row snippet to verify that the shares map correctly
preview_set = ['date', 'location', 'overall_hazard_score', 'risk_percentile', 'weather_risk_share', 'landslide_risk_share', 'risk_category']
print(master[preview_set].head(5))

 ✅ ALL PROGRAMMATIC BOUNDARY AND SHARE ADDITIVITY ASSERTIONS PASSED!
     Risk metrics and pie-chart share vectors are certified 100% sound.

📊 PORTFOLIO PREVIEW: VERIFYING DYNAMIC SHARE RATIOS
         date location  overall_hazard_score  risk_percentile  \
0  2021-01-01   Manali               20.9241            18.47   
1  2021-01-02   Manali               27.1536            60.33   
2  2021-01-03   Manali               31.7036            82.23   
3  2021-01-04   Manali               22.5166            29.13   
4  2021-01-05   Manali               31.3366            80.45   

   weather_risk_share  landslide_risk_share risk_category  
0              0.0241                0.2196           Low  
1              0.0704                0.1729      Moderate  
2              0.1587                0.1564      Moderate  
3              0.0676                0.2098           Low  
4              0.2522                0.1683      Moderate  
